In [2]:
import numpy as np
from collections import Counter

# === CONFIG ===
file_path = "/home/vitoupro/code/image_captioning/notebook/finaldata.txt"  # 📝 Your caption file
output_vocab_path = "/home/vitoupro/code/image_captioning/data/processed/vocab_char_level.json"  # 🆕 Optional output

# === Step 1: Load and clean captions ===
with open(file_path, 'r', encoding='utf-8') as f:
    raw_lines = f.readlines()

captions = []
for line in raw_lines:
    parts = line.strip().split(' ', 1)  # Split on the first space only
    if len(parts) == 2 and parts[1].strip():
        captions.append(parts[1].strip())

if not captions:
    raise ValueError("❌ No valid captions found! Make sure the file contains space-separated entries.")

# === Step 2: Tokenize each caption into characters ===
tokenized = [list(caption) for caption in captions]

# === Step 3: Build vocabulary ===
special_tokens = ['<PAD>', '<START>', '<END>', '<UNK>']
all_chars = [char for caption in tokenized for char in caption]
char_counts = Counter(all_chars)
vocab = special_tokens + sorted(char_counts.keys())

word2idx = {char: idx for idx, char in enumerate(vocab)}
idx2word = {idx: char for char, idx in word2idx.items()}

# === Step 4: Convert captions to ID sequences ===
def caption_to_ids(char_list):
    return [word2idx['<START>']] + [word2idx.get(c, word2idx['<UNK>']) for c in char_list] + [word2idx['<END>']]

sequences = [caption_to_ids(chars) for chars in tokenized]

# === Step 5: Pad sequences ===
max_len = max(len(seq) for seq in sequences)

def pad_sequence(seq, max_len):
    return seq + [word2idx['<PAD>']] * (max_len - len(seq))

padded_sequences = [pad_sequence(seq, max_len) for seq in sequences]
padded_sequences = np.array(padded_sequences)

# === Step 6: Preview results ===
print(f"✅ Total captions: {len(captions)}")
print(f"✅ Vocabulary size: {len(vocab)}")
print(f"✅ Max sequence length: {max_len}")
print("\n📌 Sample Caption:", captions[0])
print("🧩 Tokenized:", tokenized[0])
print("🔢 Sequence:", sequences[0])
print("📏 Padded:", padded_sequences[0])


✅ Total captions: 4722
✅ Vocabulary size: 70
✅ Max sequence length: 45

📌 Sample Caption: ឆ្មាភ្នែកខៀវអង្គុយលើក្រណាត់
🧩 Tokenized: ['ឆ', '្', 'ម', 'ា', 'ភ', '្', 'ន', 'ែ', 'ក', 'ខ', 'ៀ', 'វ', 'អ', 'ង', '្', 'គ', 'ុ', 'យ', 'ល', 'ើ', 'ក', '្', 'រ', 'ណ', 'ា', 'ត', '់']
🔢 Sequence: [1, 11, 67, 28, 42, 27, 67, 23, 54, 5, 6, 52, 32, 36, 9, 67, 7, 47, 29, 31, 50, 5, 67, 30, 18, 42, 19, 63, 2]
📏 Padded: [ 1 11 67 28 42 27 67 23 54  5  6 52 32 36  9 67  7 47 29 31 50  5 67 30
 18 42 19 63  2  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]


In [3]:
import json

# Save word2idx and idx2word to a file
with open("word2idx.json", "w", encoding='utf-8') as f:
    json.dump(word2idx, f, ensure_ascii=False, indent=4)

with open("idx2word.json", "w", encoding='utf-8') as f:
    json.dump(idx2word, f, ensure_ascii=False, indent=4)

print("✅ Vocabulary saved to word2idx.json and idx2word.json")


✅ Vocabulary saved to word2idx.json and idx2word.json


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader
from PIL import Image
import pandas as pd
import os
import re
import json
from sklearn.model_selection import train_test_split
import jiwer
import matplotlib.pyplot as plt

# Function to load idx2word and convert it to word2idx
def load_vocabulary(path):
    with open(path, 'r') as file:
        idx2word = json.load(file)
    word2idx = {v: int(k) for k, v in idx2word.items()}
    return idx2word, word2idx

# Load vocabulary
idx2word_path = '/home/vitoupro/code/image_captioning/notebook/idx2word.json'
idx2word, word2idx = load_vocabulary(idx2word_path)

# Encoding and decoding functions
def encode_khmer_word(word, word2idx):
    indices = []
    for character in word:
        index = word2idx.get(character)
        if index is None:
            return None, f"Character '{character}' not found in vocabulary!"
        indices.append(index)
    return indices, None

def decode_indices(indices, idx2word):
    characters = []
    for index in indices:
        character = idx2word.get(str(index))
        if character is None:
            return None, f"Index '{index}' not found in idx2word!"
        characters.append(character)
    return ''.join(characters), None

# Model Definitions (EncoderCNN and DecoderRNN)
class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(pretrained=True)
        for param in resnet.parameters():
            param.requires_grad = False
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        self.embed = nn.Linear(resnet.fc.in_features, embed_size)

    def forward(self, images):
        features = self.resnet(images)
        features = features.reshape(features.size(0), -1)
        features = self.embed(features)
        return features

class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(DecoderRNN, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.init_h = nn.Linear(hidden_size, hidden_size)  # Initialize LSTM hidden state
        self.init_c = nn.Linear(hidden_size, hidden_size)  # Initialize LSTM cell state

    def forward(self, features, captions):
        embeddings = self.embed(captions)
        h0 = self.init_h(features).unsqueeze(0).repeat(self.num_layers, 1, 1)
        c0 = self.init_c(features).unsqueeze(0).repeat(self.num_layers, 1, 1)
        lstm_out, _ = self.lstm(embeddings, (h0, c0))
        outputs = self.linear(lstm_out)
        return outputs

# Image Captioning Dataset
class ImageCaptionDataset(torch.utils.data.Dataset):
    def __init__(self, img_labels, img_dir, vocab, transform=None, max_length=50):
        self.img_labels = img_labels
        self.img_dir = img_dir
        self.vocab = vocab
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        caption = self.img_labels.iloc[idx, 1]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        indices, error = encode_khmer_word(caption, self.vocab)
        if error:
            print(f"Error encoding caption: {error}")
            indices = [self.vocab['<UNK>']] * self.max_length
        tokens = [self.vocab['<START>']] + indices + [self.vocab['<END>']]
        tokens += [self.vocab['<PAD>']] * (self.max_length - len(tokens))
        return image, torch.tensor(tokens[:self.max_length])

# Define transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
]) 

# Load dataset
annotations_file = '/home/vitoupro/code/image_captioning/notebook/finaldata.txt'
img_dir = '/home/vitoupro/code/image_captioning/notebook/downloaded_images'
# Robust custom loader to handle inconsistent spacing
data = []
with open(annotations_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            img_path, caption = parts
            data.append((img_path, caption))
        else:
            print(f"Skipping malformed line: {line}")

all_images = pd.DataFrame(data, columns=['image', 'caption'])


# Split dataset
train_images, eval_images, train_captions, eval_captions = train_test_split(
    all_images['image'].tolist(), all_images['caption'].tolist(), test_size=0.2, random_state=42
)

train_dataset = ImageCaptionDataset(
    img_labels=pd.DataFrame({'image': train_images, 'caption': train_captions}),
    img_dir=img_dir,
    vocab=word2idx,
    transform=transform,
    max_length=75
)

eval_dataset = ImageCaptionDataset(
    img_labels=pd.DataFrame({'image': eval_images, 'caption': eval_captions}),
    img_dir=img_dir,
    vocab=word2idx,
    transform=transform,
    max_length=75
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize models
encoder = EncoderCNN(embed_size=512).to(device)
decoder = DecoderRNN(embed_size=256, hidden_size=512, vocab_size=len(word2idx), num_layers=1).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])
params = list(decoder.parameters()) + list(encoder.embed.parameters())
optimizer = torch.optim.Adam(params, lr=0.001)

def custom_transform(text):
    # Lowercase the text
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Return as list of words
    return text.split()

    

def calculate_wer(gt, pred, epoch, file_path='metric.txt'):
    content_pred = ''
    content_ground_true = ''

    match_pred = re.search(r"^(.*?)<END>", pred)
    if match_pred:
        content_pred = match_pred.group(1)

    match_ground_true = re.search(r"<START>(.*?)<END>", gt)
    if match_ground_true:
        content_ground_true = match_ground_true.group(1)

    with open(file_path, 'a') as file:  # Open file in append mode
        file.write(f"Epoch {epoch}\n")
        file.write("===========================\n")
        file.write(f"pred: {content_pred}\n")
        file.write(f"true: {content_ground_true}\n")
        file.write("===========================\n")

    # Ensure non-empty
    content_ground_true = content_ground_true or ''
    content_pred = content_pred or ''
    wer_score = jiwer.wer(content_ground_true, content_pred)

    return wer_score


def calculate_cer(gt, pred):
    content_pred = ''
    content_ground_true = ''

    match_pred = re.search(r"^(.*?)<END>", pred)
    if match_pred:
        content_pred = match_pred.group(1)

    match_ground_true = re.search(r"<START>(.*?)<END>", gt)
    if match_ground_true:
        content_ground_true = match_ground_true.group(1)

    return jiwer.cer(content_ground_true or '', content_pred or '')


def evaluate_model(encoder, decoder, dataloader, device, epoch):
    encoder.eval()
    decoder.eval()
    total_wer, total_cer, num_samples = 0, 0, 0
    skipped_samples = 0
    

    with torch.no_grad():
        for batch_idx, (images, captions) in enumerate(dataloader):
            images, captions = images.to(device), captions.to(device)
            features = encoder(images)
            outputs = decoder(features, captions[:, :-1])
            predicted_captions = outputs.argmax(-1)

            for i in range(len(captions)):
                gt_token_ids = captions[i].tolist()
                pred_token_ids = predicted_captions[i].tolist()

                gt_caption, gt_err = decode_indices(gt_token_ids, idx2word)
                pred_caption, pred_err = decode_indices(pred_token_ids, idx2word)

                if not gt_caption or not pred_caption:
                    print(f"\n[⚠️ Skipped Sample] Batch {batch_idx}, Index {i}, Epoch {epoch}")
                    print(f"  🔢 GT token IDs     : {gt_token_ids}")
                    print(f"  🔢 Pred token IDs   : {pred_token_ids}")
                    print(f"  ❌ GT decode error  : {gt_err}")
                    print(f"  ❌ Pred decode error: {pred_err}")
                    skipped_samples += 1
                    continue

                try:
                    wer = calculate_wer(gt_caption, pred_caption, epoch)
                    cer = calculate_cer(gt_caption, pred_caption)
                    
    
                except Exception as e:
                    print(f"\n[‼️ WER/CER Error] Sample {i} failed:")
                    print(f"  GT Caption : {gt_caption}")
                    print(f"  Pred Caption : {pred_caption}")
                    print(f"  Error : {e}")
                    skipped_samples += 1
                    continue

                total_wer += wer
                total_cer += cer
                num_samples += 1

    if num_samples == 0:
        print("⚠️ No valid samples evaluated.")
        return 1.0, 1.0

    avg_wer = total_wer / num_samples
    avg_cer = total_cer / num_samples
    
    print(f"\n✅ Epoch {epoch} Evaluation Summary:")
    print(f"   Avg WER  : {avg_wer:.3f}")
    print(f"   Avg CER  : {avg_cer:.3f}")
    print(f"   Skipped  : {skipped_samples} samples")
    
    return avg_wer, avg_cer


/home/vitoupro/code/image_captioning/image_env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/vitoupro/code/image_captioning/image_env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
print("✅ Encoder Type:", type(encoder)) 
print("✅ Number of Parameters in Encoder:", sum(p.numel() for p in encoder.parameters()))
print("✅ Number of Trainable Parameters:", sum(p.numel() for p in encoder.parameters() if p.requires_grad))

✅ Encoder Type: <class '__main__.EncoderCNN'>
✅ Number of Parameters in Encoder: 24557120
✅ Number of Trainable Parameters: 1049088


In [5]:
print("✅ Decoder Type:", type(decoder))
print("✅ Number of Parameters in Decoder:", sum(p.numel() for p in decoder.parameters()))
print("✅ Number of Trainable Parameters:", sum(p.numel() for p in decoder.parameters() if p.requires_grad))


✅ Decoder Type: <class '__main__.DecoderRNN'>
✅ Number of Parameters in Decoder: 2157640
✅ Number of Trainable Parameters: 2157640


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader
from PIL import Image
import pandas as pd
import os
import re
import json
from sklearn.model_selection import train_test_split
import jiwer
import matplotlib.pyplot as plt

# Function to load idx2word and convert it to word2idx
def load_vocabulary(path):
    with open(path, 'r') as file:
        idx2word = json.load(file)
    word2idx = {v: int(k) for k, v in idx2word.items()}
    return idx2word, word2idx

# Load vocabulary
idx2word_path = '/home/vitoupro/code/image_captioning/notebook/idx2word.json'
idx2word, word2idx = load_vocabulary(idx2word_path)

# Encoding and decoding functions
def encode_khmer_word(word, word2idx):
    indices = []
    for character in word:
        index = word2idx.get(character)
        if index is None:
            return None, f"Character '{character}' not found in vocabulary!"
        indices.append(index)
    return indices, None

def decode_indices(indices, idx2word):
    characters = []
    for index in indices:
        character = idx2word.get(str(index))
        if character is None:
            return None, f"Index '{index}' not found in idx2word!"
        characters.append(character)
    return ''.join(characters), None

# Attention Module
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super(Attention, self).__init__()
        self.attn = nn.Linear(encoder_dim + decoder_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1)

    def forward(self, encoder_out, hidden):
        hidden = hidden.unsqueeze(1).repeat(1, encoder_out.size(1), 1)
        attn_input = torch.cat((encoder_out, hidden), dim=2)
        energy = torch.tanh(self.attn(attn_input))
        attention = self.v(energy).squeeze(2)
        alpha = torch.softmax(attention, dim=1)
        context = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)
        return context, alpha

class EncoderCNN(nn.Module):
    def __init__(self):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(pretrained=True)
        for name, param in resnet.named_parameters():
            param.requires_grad = 'layer4' in name
        modules = list(resnet.children())[:-2]  # Keep conv feature map
        self.resnet = nn.Sequential(*modules)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((14, 14))

    def forward(self, images):
        features = self.resnet(images)  # (B, 2048, H, W)
        features = self.adaptive_pool(features)
        features = features.permute(0, 2, 3, 1)  # (B, 14, 14, 2048)
        features = features.view(features.size(0), -1, features.size(-1))  # (B, 196, 2048)
        return features
    
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, attention_dim=256, encoder_dim=2048, num_layers=1, dropout_prob=0.3):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.attention = Attention(encoder_dim, hidden_size, attention_dim)
        self.lstm = nn.LSTMCell(embed_size + encoder_dim, hidden_size)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout_prob)
        self.init_h = nn.Linear(encoder_dim, hidden_size)
        self.init_c = nn.Linear(encoder_dim, hidden_size)

    def forward(self, encoder_out, captions):
        batch_size = encoder_out.size(0)
        seq_len = captions.size(1)
        vocab_size = self.linear.out_features

        # Init hidden state and cell state
        h = self.init_h(encoder_out.mean(dim=1))  # (B, hidden_size)
        c = self.init_c(encoder_out.mean(dim=1))

        outputs = torch.zeros(batch_size, seq_len - 1, vocab_size).to(encoder_out.device)

        for t in range(seq_len - 1):
            embeddings = self.dropout(self.embed(captions[:, t]))  # (B, embed_size)
            context, _ = self.attention(encoder_out, h)            # (B, encoder_dim)
            lstm_input = torch.cat([embeddings, context], dim=1)   # (B, embed + encoder_dim)
            h, c = self.lstm(lstm_input, (h, c))                   # LSTMCell
            output = self.linear(self.dropout(h))                 # (B, vocab_size)
            outputs[:, t, :] = output

        return outputs

# Image Captioning Dataset
class ImageCaptionDataset(torch.utils.data.Dataset):
    def __init__(self, img_labels, img_dir, vocab, transform=None, max_length=50):
        self.img_labels = img_labels
        self.img_dir = img_dir
        self.vocab = vocab
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        caption = self.img_labels.iloc[idx, 1]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        indices, error = encode_khmer_word(caption, self.vocab)
        if error:
            print(f"Error encoding caption: {error}")
            indices = [self.vocab['<UNK>']] * self.max_length
        tokens = [self.vocab['<START>']] + indices + [self.vocab['<END>']]
        tokens += [self.vocab['<PAD>']] * (self.max_length - len(tokens))
        return image, torch.tensor(tokens[:self.max_length])

# Define transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
]) 

# Load dataset
annotations_file = '/home/vitoupro/code/image_captioning/notebook/finaldata.txt'
img_dir = '/home/vitoupro/code/image_captioning/notebook/downloaded_images'
# Robust custom loader to handle inconsistent spacing
data = []
with open(annotations_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            img_path, caption = parts
            data.append((img_path, caption))
        else:
            print(f"Skipping malformed line: {line}")

all_images = pd.DataFrame(data, columns=['image', 'caption'])


# Split dataset
train_images, eval_images, train_captions, eval_captions = train_test_split(
    all_images['image'].tolist(), all_images['caption'].tolist(), test_size=0.2, random_state=42
)

train_dataset = ImageCaptionDataset(
    img_labels=pd.DataFrame({'image': train_images, 'caption': train_captions}),
    img_dir=img_dir,
    vocab=word2idx,
    transform=transform,
    max_length=50
)

eval_dataset = ImageCaptionDataset(
    img_labels=pd.DataFrame({'image': eval_images, 'caption': eval_captions}),
    img_dir=img_dir,
    vocab=word2idx,
    transform=transform,
    max_length=50
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=16, shuffle=False)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize models
encoder = EncoderCNN().to(device)
decoder = DecoderRNN(embed_size=256, hidden_size=512, vocab_size=len(word2idx), num_layers=1,dropout_prob=0.3).to(device)

# Loss and optimizer 
criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])
params = list(decoder.parameters()) + list(encoder.parameters())
optimizer = torch.optim.Adam(params, lr=0.001)

def custom_transform(text):
    # Lowercase the text
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Return as list of words
    return text.split()

    

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import re
import jiwer

smoothing = SmoothingFunction().method4  # More stable smoothing

def calculate_wer(gt, pred, epoch, file_path='metric.txt'):
    match_pred = re.search(r"^(.*?)<END>", pred)
    content_pred = match_pred.group(1) if match_pred else ""

    match_gt = re.search(r"<START>(.*?)<END>", gt)
    content_gt = match_gt.group(1) if match_gt else ""

    with open(file_path, 'a') as file:
        file.write(f"Epoch {epoch}\n")
        file.write("===========================\n")
        file.write(f"pred: {content_pred}\n")
        file.write(f"true: {content_gt}\n")
        file.write("===========================\n")

    return jiwer.wer(content_gt or '', content_pred or '')

def calculate_cer(gt, pred):
    match_pred = re.search(r"^(.*?)<END>", pred)
    content_pred = match_pred.group(1) if match_pred else ""

    match_gt = re.search(r"<START>(.*?)<END>", gt)
    content_gt = match_gt.group(1) if match_gt else ""

    return jiwer.cer(content_gt or '', content_pred or '')

def evaluate_model(encoder, decoder, dataloader, device, epoch):
    encoder.eval()
    decoder.eval()

    total_wer, total_cer = 0, 0
    num_samples = 0
    skipped = 0

    all_references = []
    all_predictions = []

    with torch.no_grad():
        for batch_idx, (images, captions) in enumerate(dataloader):
            images, captions = images.to(device), captions.to(device)
            features = encoder(images)
            outputs = decoder(features, captions[:, :-1])  # Predict T-1 tokens
            predicted_captions = outputs.argmax(-1)

            for i in range(captions.size(0)):
                gt_ids = captions[i].tolist()
                pred_ids = predicted_captions[i].tolist()

                gt_text, gt_err = decode_indices(gt_ids, idx2word)
                pred_text, pred_err = decode_indices(pred_ids, idx2word)

                if not gt_text or not pred_text or gt_err or pred_err:
                    skipped += 1
                    continue

                # Extract only content between <START> and <END>
                ref_match = re.search(r"<START>(.*?)<END>", gt_text)
                pred_match = re.search(r"^(.*?)<END>", pred_text)
                reference = ref_match.group(1).strip() if ref_match else ""
                prediction = pred_match.group(1).strip() if pred_match else ""

                if not reference or not prediction:
                    continue

                wer = calculate_wer(gt_text, pred_text, epoch)
                cer = calculate_cer(gt_text, pred_text)
                total_wer += wer
                total_cer += cer
                num_samples += 1

                # Use character-level tokens for BLEU
                ref_chars = list(reference)
                pred_chars = list(prediction)

                if len(ref_chars) == 0 or len(pred_chars) == 0:
                    continue

                all_references.append([ref_chars])  # list of one reference
                all_predictions.append(pred_chars)

    # Corpus-level BLEU
    # Corpus-level BLEU
    bleu1 = corpus_bleu(all_references, all_predictions, weights=(1, 0, 0, 0), smoothing_function=smoothing)
    bleu2 = corpus_bleu(all_references, all_predictions, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
    bleu3 = corpus_bleu(all_references, all_predictions, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothing)
    bleu4 = corpus_bleu(all_references, all_predictions, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)


    avg_wer = total_wer / num_samples if num_samples else 1.0
    avg_cer = total_cer / num_samples if num_samples else 1.0

    print(f"\n✅ Epoch {epoch} Evaluation Summary:")
    print(f"   WER     : {avg_wer:.3f}")
    print(f"   CER     : {avg_cer:.3f}")
    print(f"   BLEU-1  : {bleu1:.3f}")
    print(f"   BLEU-2  : {bleu2:.3f}")
    print(f"   BLEU-3  : {bleu3:.3f}")
    print(f"   BLEU-4  : {bleu4:.3f}")
    print(f"   Skipped : {skipped} samples")

    return avg_wer, avg_cer, bleu1, bleu2,bleu3, bleu4

/home/vitoupro/code/image_captioning/image_env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/vitoupro/code/image_captioning/image_env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
print("✅ Encoder Type:", type(encoder))
print("✅ Number of Parameters in Encoder:", sum(p.numel() for p in encoder.parameters()))
print("✅ Number of Trainable Parameters:", sum(p.numel() for p in encoder.parameters() if p.requires_grad))

✅ Encoder Type: <class '__main__.EncoderCNN'>
✅ Number of Parameters in Encoder: 23508032
✅ Number of Trainable Parameters: 14964736


In [6]:
print("✅ Decoder Type:", type(decoder))
print("✅ Number of Parameters in Decoder:", sum(p.numel() for p in decoder.parameters()))
print("✅ Number of Trainable Parameters:", sum(p.numel() for p in decoder.parameters() if p.requires_grad))


✅ Decoder Type: <class '__main__.DecoderRNN'>
✅ Number of Parameters in Decoder: 8579143
✅ Number of Trainable Parameters: 8579143


In [ ]:
import matplotlib.pyplot as plt

# Training Loop
num_epochs = 30
best_wer = float('inf')

# Initialize tracking lists
train_losses = []
val_wers = []
val_cers = []


for epoch in range(num_epochs):
    encoder.train()
    decoder.train()
    total_loss = 0
    for images, captions in train_loader:
        images, captions = images.to(device), captions.to(device)
        features = encoder(images)
        outputs = decoder(features, captions[:, :-1])
        loss = criterion(outputs.view(-1, len(word2idx)), captions[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f'Epoch {epoch+1}: Train Loss: {total_loss/len(train_loader)}')
    wer, cer = evaluate_model(encoder, decoder, eval_loader, device, epoch)
    val_wers.append(wer)
    val_cers.append(cer)
    
    

    if wer < best_wer:
        best_wer = wer
        
# 📊 Plotting all tracked metrics
epochs = list(range(1, num_epochs + 1))
plt.figure(figsize=(10, 6))

plt.plot(epochs, train_losses, label='Train Loss', marker='o')
plt.plot(epochs, val_wers, label='WER (↓)', marker='s')
plt.plot(epochs, val_cers, label='CER (↓)', marker='^')

plt.xlabel('Epoch')
plt.ylabel('Metric')
plt.title('Training Progress (Loss, WER, CER)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("training_progress.png")
plt.show()


Epoch 1: Train Loss: 0.37319012491379755

✅ Epoch 0 Evaluation Summary:
   Avg WER  : 0.990
   Avg CER  : 0.336
   Skipped  : 0 samples
Epoch 2: Train Loss: 0.35764710781937936

✅ Epoch 1 Evaluation Summary:
   Avg WER  : 0.995
   Avg CER  : 0.363
   Skipped  : 0 samples
Epoch 3: Train Loss: 0.3437652337854191

✅ Epoch 2 Evaluation Summary:
   Avg WER  : 0.992
   Avg CER  : 0.337
   Skipped  : 0 samples
Epoch 4: Train Loss: 0.3296216332306296

✅ Epoch 3 Evaluation Summary:
   Avg WER  : 0.995
   Avg CER  : 0.342
   Skipped  : 0 samples
Epoch 5: Train Loss: 0.31588999598713247

✅ Epoch 4 Evaluation Summary:
   Avg WER  : 0.992
   Avg CER  : 0.334
   Skipped  : 0 samples
Epoch 6: Train Loss: 0.3032610954369529

✅ Epoch 5 Evaluation Summary:
   Avg WER  : 0.995
   Avg CER  : 0.337
   Skipped  : 0 samples
Epoch 7: Train Loss: 0.2941336558531907

✅ Epoch 6 Evaluation Summary:
   Avg WER  : 0.992
   Avg CER  : 0.322
   Skipped  : 0 samples
Epoch 8: Train Loss: 0.2854379630189831

✅ Epoch 7 E

In [2]:
def predict_caption(image_path, encoder, decoder, transform, device, idx2word, word2idx):
    encoder.eval()
    decoder.eval()
    
    # Load and transform the image
    image = Image.open(image_path).convert('RGB')
    if transform:
        image = transform(image)
    image = image.unsqueeze(0).to(device)  # Add batch dimension and transfer to device
    
    # Generate features from the image using the encoder
    features = encoder(image)
    
    # Start the sequence with the <START> token
    predicted_indices = [word2idx['<START>']]
    predictions = []
    
    # Initial input to the LSTM is the <START> token
    input_idx = torch.tensor([predicted_indices[-1]], dtype=torch.long).to(device)
    
    # Initialize the LSTM state
    h, c = None, None
    
    # Generate words until the <END> token is predicted or the max length is reached
    for _ in range(75):  # Assuming max length of 20 for safety
        input_idx = input_idx.unsqueeze(0)  # Add batch dimension for single time-step prediction
        if h is None and c is None:
            # Generate initial hidden states from features
            h = decoder.init_h(features).unsqueeze(0).repeat(decoder.num_layers, 1, 1)
            c = decoder.init_c(features).unsqueeze(0).repeat(decoder.num_layers, 1, 1)
        
        outputs, (h, c) = decoder.lstm(decoder.embed(input_idx), (h, c))
        outputs = decoder.linear(outputs.squeeze(1))
        
        # Get the predicted word index
        predicted_index = outputs.argmax(-1).item()
        predicted_indices.append(predicted_index)
        predictions.append(idx2word[str(predicted_index)])  # Decode to word
        
        # Prepare the next input
        input_idx = torch.tensor([predicted_index], dtype=torch.long).to(device)
        
        # Stop if the <END> token is predicted
        if predicted_index == word2idx['<END>']:
            break
    
    predicted_caption = ' '.join(predictions)  # Join the predicted words
    
    return predicted_caption

# Example usage
image_path = '/home/vitoupro/code/image_captioning/data/pexels_63.jpg'
predicted_caption = predict_caption(image_path, encoder, decoder, transform, device, idx2word, word2idx)
print("Predicted Caption:", predicted_caption.replace(" ", ""))


NameError: name 'encoder' is not defined

In [26]:
# Save the encoder and decoder models
torch.save(encoder.state_dict(), 'encoder_finalv1.pth')
torch.save(decoder.state_dict(), 'decoder_finalv1.pth')

In [27]:
# Save the model state dictionary
torch.save({
    'encoder_state_dict': encoder.state_dict(),
    'decoder_state_dict': decoder.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'captioning_model_finalv1.pth')


In [1]:
!which python

/home/vitoupro/code/image_captioning/image_env/bin/python
